# Research 03 — PPO/GRPO loss：clip、advantage、KL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/research/03-ppo-grpo-loss.ipynb)

- **日期**：2026-10-02
- **定位**：Research PyTorch 轮次的 RL 算法面。DPO 是离线偏好优化，PPO/GRPO 是在线 RL —— 这一轮要能手写任意一个 loss 并解释每一项的来历。
- **环境**：纯 PyTorch，CPU 即可。

配套笔记：[03-ppo-grpo-loss.md](03-ppo-grpo-loss.md)


## 今日目标

1. 手写 PPO 的 clipped loss，并数值验证 clip 的"可信域外梯度置零"行为。
2. 手写 GAE，验证 $ \lambda=0 $ 和 $ \lambda=1 $ 两个极端。
3. 手写 GRPO 的组内 advantage + KL，用 toy bandit 训出可见的策略改进。

记号：$ \rho_t = \pi_\theta(a_t|s_t) / \pi_{old}(a_t|s_t) $ ，$ A_t $ advantage，$ \epsilon $ clip 幅度（常用 0.2）。


## 1. PPO clipped loss

$$ \mathcal{L}^{clip} = \mathbb{E}\left[\min\left(\rho_t A_t, \mathrm{clip}(\rho_t, 1-\epsilon, 1+\epsilon) A_t\right)\right] $$

$ \min $ 取悲观估计：$ A>0 $ 时防止一步冲太远，$ A<0 $ 时防止把坏动作的概率降过头。$ \rho $ 跑出区间后，那一项的梯度为零 —— 不是把 ratio 拉回来，是拒绝在可信域外优化。


In [ ]:
import torch

def ppo_loss(logp_new, logp_old, advantages, clip_eps=0.2):
    """PPO 的 clipped surrogate loss。

    入参（三个张量形状必须一致，(B,) 序列级或 (B, T) token 级）:
      logp_new: 当前策略 πθ 对已采样动作的 log 概率（带梯度）
      logp_old: 采样时的行为策略对同一批动作的 log 概率（不带梯度，当常数）
      advantages: 同形状的 advantage（已 detach，梯度只经 logp_new 回传）
      clip_eps: trust region 半宽（默认 0.2，ratio 被夹在 [0.8, 1.2]）
    返回: 标量 loss（最大化目标取负，交给 optimizer 最小化）。"""
    ratio = torch.exp(logp_new - logp_old)                    # ← log 空间相减后 exp
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages
    return -torch.min(surr1, surr2).mean()                    # ← 最大化目标 → 取负

# 检查 1：policy 未动（ratio=1）时 loss 应等于 -mean(A)
A = torch.tensor([1.0, -1.0, 2.0])
lp = torch.tensor([-0.5, -1.5, -2.0])
print('检查 1: ratio=1 时 loss =', round(ppo_loss(lp, lp, A).item(), 4),
      ' 期望 -mean(A) =', round(-A.mean().item(), 4))

# 检查 2：A>0 且 ratio 冲出上界 → 该项梯度应为 0（可信域外拒绝优化）
lp_new = torch.tensor([-0.5 + 0.9], requires_grad=True)   # ratio = e^0.9 ≈ 2.46 >> 1.2
ppo_loss(lp_new, torch.tensor([-0.5]), torch.tensor([1.0]), 0.2).backward()
print('检查 2: 冲出 clip 上界后梯度 =', lp_new.grad.item(), '（应为 0）')

# 检查 3：A<0 且 ratio 已在下界内侧 → min 取悲观项，梯度正常回传
lp_new = torch.tensor([-0.5], requires_grad=True)          # ratio = 1
ppo_loss(lp_new, torch.tensor([-0.5]), torch.tensor([-1.0]), 0.2).backward()
print('检查 3: 区间内梯度 =', round(lp_new.grad.item(), 3), '（非零，正常回传）')


## 2. GAE：advantage 的估计器

TD 残差 $ \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t) $ ，$ A^{GAE}_t = \sum_l (\gamma\lambda)^l \delta_{t+l} $ 。反向递推实现。


In [ ]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    """GAE(λ)：把 TD 残差 δ_t 按 (γλ)^k 衰减加权求和。

    rewards: (T,) 每步的 reward；
    values: (T+1,) critic 的 V(s_t) —— 比 rewards 多一位，最后一位是 bootstrap
            的 V(s_T)（序列被截断时用它续命，自然终止填 0）；
    gamma: 折扣 γ；lam: λ，0 = 纯 TD、1 = 蒙特卡洛（下面两个极端自检各验一头）。
    返回: (T,) 每一步的 advantage A_t。"""
    adv = torch.zeros_like(rewards)
    gae = 0.0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        gae = delta + gamma * lam * gae
        adv[t] = gae
    return adv

r = torch.tensor([1.0, 0.0, 0.0, 1.0])
v0 = torch.zeros(5)
print('λ=0（纯 TD）:', [round(x, 2) for x in compute_gae(r, v0, 1.0, 0.0).tolist()],
      ' 期望=δ 本身:', [1.0, 0.0, 0.0, 1.0])
print('λ=1, γ=1, V=0（蒙特卡洛）:', [round(x, 2) for x in compute_gae(r, v0, 1.0, 1.0).tolist()],
      ' 期望=未来 reward 之和:', [2.0, 1.0, 1.0, 1.0])


## 3. GRPO：组内相对 advantage

同一 prompt 采样 $ G $ 个回答，$ A_i = \frac{R_i - \mathrm{mean}(R_{1..G})}{\mathrm{std}(R_{1..G})} $ 。组内 advantage 之和恒为 0 —— 梯度只推"比同组平均好"的样本。下面用 toy bandit 把整个闭环跑一遍：策略是 6 个动作上的 logits，目标是学会选动作 3。


In [ ]:
import torch.nn.functional as F

def grpo_advantages(rewards):
    """GRPO 的组内标准化 advantage（用同组样本当基线，省掉 critic）。

    rewards: (P, G) —— P 个 prompt，每个 prompt 采样 G 个 completion 得到的 reward。
    返回: (P, G)，每组内 (r − mean) / std。同一 prompt 的样本互为基线。"""
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True)
    return (rewards - mean) / (std + 1e-8)     # Dr. GRPO 主张去掉 std 项防难度偏置

torch.manual_seed(0)
P, G, K, TARGET = 16, 8, 6, 3
logits = torch.zeros(K, requires_grad=True)          # 策略：动作上的 logits
ref_logp = torch.log_softmax(torch.zeros(K), -1)     # ref = 均匀分布
opt = torch.optim.Adam([logits], lr=0.1)

for step in range(200):
    logp = torch.log_softmax(logits, -1)
    probs = logp.exp()
    actions = torch.multinomial(probs, P * G, replacement=True).view(P, G)
    rewards = (actions == TARGET).float()
    adv = grpo_advantages(rewards)
    assert adv.sum().abs() < 1e-4, '组内 advantage 之和应为 0'

    # 策略梯度项：-logp(a) * A；KL 项（对 ref 的解析 KL，拴住别跑偏）
    pg = -(logp[actions] * adv).mean()
    kl = (probs * (logp - ref_logp)).sum()
    loss = pg + 0.01 * kl
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 40 == 0:
        print(f'step {step:3d}  P(动作3) {probs[TARGET].item():.3f}  loss {loss.item():.4f}')
print('\n最终动作分布:', [round(x, 3) for x in probs.tolist()], '（目标动作是 3）')


## 4. KL 估计器三兄弟（实现里最容易写错符号的地方）

采样 $ a \sim \pi_\theta $ ，记 $ r = \pi_{ref}(a)/\pi_\theta(a) $ ：

- $ k_1 = -\log r = \log\pi_\theta - \log\pi_{ref} $ ：无偏、高方差、单样本可为负
- $ k_2 = \frac{1}{2}(\log\pi_\theta - \log\pi_{ref})^2 $ ：低方差、有偏
- $ k_3 = (r - 1) - \log r $ ：无偏、恒非负，实现里最常见


In [ ]:
def kl_estimators(logp, ref_logp):
    """三种 KL 估计量，输入是同一批采样 token 上的 log 概率。

    logp: (N,) 样本在当前策略 πθ 下的 log p（样本本身从 πθ 采的）；
    ref_logp: (N,) 同一采样在 ref 模型下的 log p。
    返回: (k1, k2, k3)，各 (N,)；对 batch 取均值即 KL(πθ || πref) 的估计。"""
    r = torch.exp(ref_logp - logp)
    k1 = logp - ref_logp
    k2 = 0.5 * (logp - ref_logp) ** 2
    k3 = (r - 1) - (ref_logp - logp)
    return k1, k2, k3

# 数值自检：从 π_θ 采样，三个估计器的均值都应逼近 KL(π_θ || π_ref)
torch.manual_seed(0)
logits = torch.tensor([2.0, 1.0, 0.0, -1.0])
ref_logits = torch.zeros(4)
lp, rlp = torch.log_softmax(logits, -1), torch.log_softmax(ref_logits, -1)
true_kl = (lp.exp() * (lp - rlp)).sum()
samples = torch.multinomial(lp.exp(), 20000, replacement=True)
k1, k2, k3 = kl_estimators(lp[samples], rlp[samples])
print(f'真实 KL = {true_kl.item():.4f} | k1 均值 {k1.mean().item():.4f}（可为负: {(k1<0).float().mean().item():.2f}）'
      f' | k2 均值 {k2.mean().item():.4f} | k3 均值 {k3.mean().item():.4f}（恒非负: {(k3>=0).all().item()}）')


## 5. Debug 演练：五个可跑的 bug 现场

清单背一百遍不如现场看一次症状。下面五个 bug 都能跑、都有指纹，修复都写在各自正下方。

In [ ]:
# BUG 1：log-ratio 忘做 exp，直接当 ratio
def ppo_loss_bug1(logp_new, logp_old, advantages, clip_eps=0.2):
    ratio = logp_new - logp_old                       # ← BUG：这是 log 空间的差，不是 ratio
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages
    return -torch.min(surr1, surr2).mean()

torch.manual_seed(0)
logp_old = torch.tensor([-1.2, -0.7, -2.0, -0.4])
logp_new = logp_old.clone().requires_grad_(True)      # 新旧策略相同：ratio 处处应为 1
adv = torch.tensor([1.0, -0.5, 0.8, -1.2])
lb = ppo_loss_bug1(logp_new, logp_old, adv); lb.backward()
print('BUG 1 症状: 同策略时 loss =', round(lb.item(), 4), '（正确自检应为 -mean(A) =', round(-adv.mean().item(), 4), '）—— 对不上就是它')
print('         再看一行: Δlogp = -0.5 时 buggy 的"ratio" = -0.5（概率比不可能为负），真 ratio = e^-0.5 = 0.607')

**修复 1**：ratio 必须先 exp 回概率空间：`ratio = torch.exp(logp_new - logp_old)`（第 1 节的正确版就是）。

**为什么这条 bug 阴险**：它不报错也不 NaN，同策略时仍有梯度在流，只是 surrogate 的几何全错 —— 把 Δlogp 本人当 ratio 之后，ratio 可以是负数、clip 的 [0.8, 1.2] 区间套在对数差上完全没意义，策略越训越偏。抓它的办法是同策略自检：新旧策略相同时 ratio 应恒为 1、loss 应等于 -mean(A)；对不上，先查 exp 有没有漏。

In [ ]:
# BUG 2：clip 上下界写反
def ppo_loss_bug2(logp_new, logp_old, advantages, clip_eps=0.2):
    ratio = torch.exp(logp_new - logp_old)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 + clip_eps, 1 - clip_eps) * advantages   # ← BUG：min > max
    return -torch.min(surr1, surr2).mean()

logp_new2 = logp_old.clone().requires_grad_(True)
lb2 = ppo_loss_bug2(logp_new2, logp_old, adv)
print('BUG 2 症状: 同策略时 loss =', round(lb2.item(), 4), '（正确 -mean(A) =', round(-adv.mean().item(), 4), '）；clamp(ratio, 1.2, 0.8) 恒等于 0.8 —— ratio 被钉死')

**修复 2**：上下界别写反：`torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)`。

**为什么有效**：`torch.clamp` 在 min > max 时不报错、直接恒返回 max —— ratio 被钉在 0.8，trust region 形同虚设，症状只在数值对账时现形。写完 clip 后固定做一遍三点自检（第 1 节检查 1–3 就是）：同策略 loss = -mean(A)、冲出上界梯度为 0、区间内梯度非零。

In [ ]:
# BUG 3：advantage 没 detach（从 critic 算出来后直接用）
theta = torch.zeros(6, requires_grad=True)                       # 玩具 policy 的 logits
rewards3 = torch.tensor([0., 0., 0., 1., 0., 0.])
logp3 = torch.log_softmax(theta, -1)
V_fake = (torch.softmax(theta, -1) * rewards3).sum()             # 假装 critic 输出（和 policy 共享参数）
adv_bug = rewards3 - V_fake                                      # ← BUG：带着对 theta 的梯度
loss_bug = -(logp3 * adv_bug).sum(); loss_bug.backward()
g_bug = theta.grad.clone()

theta2 = torch.zeros(6, requires_grad=True)
logp3b = torch.log_softmax(theta2, -1)
adv_ok = rewards3 - (torch.softmax(theta2, -1) * rewards3).sum().detach()
loss_ok = -(logp3b * adv_ok).sum(); loss_ok.backward()
print('BUG 3 症状: 带梯度版 vs detach 版的 policy 梯度最大差 =', round((g_bug - theta2.grad).abs().max().item(), 4), '（≠0 = 更新方向被 advantage 路径污染）')

**修复 3**：advantage 进 loss 前一律 detach：`advantages = advantages.detach()`，或者构造时就 `with torch.no_grad():` 算。

**为什么有效**：advantage 在 PPO/GRPO 里是"给每个动作打的分数"，是常数权重不是优化对象。不 detach 时，梯度会沿 advantage 的计算图反向流回 critic（甚至像上面这样流回 policy 本身），policy 被推着去改"评分标准"而不是改行为 —— 真实框架里还常伴随 retain_graph 报错或显存莫名上涨。口诀：**采样侧产出的一切量（logp_old、advantage、reward）进 learner 时全部当常数。**

In [ ]:
# BUG 4：GRPO 分组 std 没加 eps，全同 reward 组出 NaN
def grpo_bug(rewards):
    return (rewards - rewards.mean(dim=1, keepdim=True)) / rewards.std(dim=1, keepdim=True)  # ← BUG：没 +1e-8

rw = torch.tensor([[1., 1., 1., 1.],      # 这个 prompt 组全对：std = 0
                   [0., 1., 0., 1.]])
print('BUG 4 症状: buggy advantage =', grpo_bug(rw).tolist()[0], '（0/0 = NaN，下一步 loss 跟着 NaN）')
print('         正确版 advantage =', grpo_advantages(rw).tolist()[0], '（全同组本就该全 0：组内没有相对信号）')

**修复 4**：分母加 eps：`(r - mean) / (std + 1e-8)`（第 3 节的正确版就是这么写的）。

**为什么有效**：一组样本 reward 全同时，组内相对信号本来就是零，advantage 全 0 是正确答案；NaN 纯粹是 0/0 的实现事故。这类 NaN 有个典型指纹：训练跑了很久才突然出现，因为要等某个 prompt 组恰好全对或全错（简单样本学会之后全对组会越来越多）。Dr. GRPO 索性主张直接去掉 std 项，也是绕开这个坑的一种走法。

In [ ]:
# BUG 5：KL 项符号写反（惩罚变奖励）
torch.manual_seed(0)
logits_p = torch.tensor([2.0, 1.0, 0.0, -1.0])
logits_r = torch.tensor([1.0, 1.0, 1.0, 0.0])
lp  = torch.log_softmax(logits_p, -1); lr_ = torch.log_softmax(logits_r, -1)
probs = torch.softmax(logits_p, -1)
sample_idx = torch.multinomial(probs, 20000, replacement=True)
_, _, k3 = kl_estimators(lp[sample_idx], lr_[sample_idx])
print('BUG 5 症状: 正确 KL 估计 =', round(k3.mean().item(), 3), '；写反符号后进 loss 的"KL" =', round(-k3.mean().item(), 3), '（KL 恒非负，负数就是符号反了）')

**修复 5**：KL 进 loss 的符号先用一句话验：KL(πθ‖πref) ≥ 0 恒成立，**打印出来是负数先查符号**，别急着调 β。GRPO 的 loss 里 KL 是 `+ beta * KL`（惩罚），reward 里折 KL 是 `r - beta * KL`（第 4 节的估计器对拍真值就是为这一刻准备的）。

## 6. 三方对照 + debug 速查

| | DPO | PPO | GRPO |
|---|---|---|---|
| 数据 | 离线偏好对 | 在线采样 | 在线采样（组内） |
| 网络 | policy + ref | policy/ref/reward/critic | policy + ref |
| advantage | 隐式（log-ratio 差） | GAE + critic | 组内标准化 |
| KL 位置 | 闭式解里 | reward 里或 loss 里 | loss 里显式项 |

**现场 debug 顺序**（修复都在第 5 节各 bug 正下方，这里只留顺序）：同策略 ratio 自检（BUG 1）→ clip 三点对账（BUG 2）→ 采样侧量全部 detach（BUG 3）→ 全同 reward 组防 NaN（BUG 4）→ KL 打印值非负（BUG 5）。另外两条没有现场版、记住即可：prompt token 漏进 loss 就回 Research 02 看 mask；advantage 广播维度错位先打印它和 logp 的 shape 对照。

## 小结

- PPO 的命门：ratio 的 log 空间计算、clip 的 min 悲观语义、可信域外梯度置零。
- GAE 是"用 V 把未来折现回来"，两个极端 λ=0/λ=1 必须能脱口而出。
- GRPO 用组内名次换掉 critic：组内 advantage 和为 0、std 位置（组内不是全体）是最容易错的两点。
- KL 符号方向写反是静默杀手：训完离 ref 越来越远，loss 还在降。

**下一篇**：Research 04 — 注意力机制手写（SDPA、多头、causal mask 与 KV cache）
